In [3]:
import numpy as np
import itertools

In [4]:
class PolynomialRegression:
    def __init__(self, degree=1):
        self.degree = degree
        self.weights = None
        self.mean = None
        self.std = None
        self.n_features = None
        self.feature_combinations = []

    def fit_scaler(self, X):
        means = []
        stds  = []
        for i in range(len(X.T)):
            means.append(X.T[i].mean())
            stds.append(X.T[i].std(axis=0)) if X.T[i].std(axis=0) != 0 else stds.append(1)
        self.mean = means
        self.std = stds

    def transform_scaler(self, X):
        return (X - self.mean) / self.std

    def create_poly_features(self, X):
        self.feature_combinations = []
        indices = list(range(self.n_features))
        columns = []
        for d in range(1, self.degree + 1):
            combinations = itertools.combinations_with_replacement(indices, d)
            for comb in combinations:
                columns.append(np.prod(X[:, list(comb)], axis=1))
                self.feature_combinations.append(comb)
        return np.column_stack(columns)

    def add_bias(self, X):
        return np.column_stack([np.ones(len(X)), X])

    def fit(self, X, y):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        y = np.asarray(y)
        self.n_features = X.shape[1]
        self.fit_scaler(X)
        X_scaled = self.transform_scaler(X)
        X_poly = self.create_poly_features(X_scaled)
        X_full = self.add_bias(X_poly)
        self.weights = np.linalg.lstsq(X_full, y, rcond=None)[0]
        return self

    def predict(self, X):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        if X.shape[1] != self.n_features:
            raise ValueError
        X_scaled = self.transform_scaler(X)
        columns = []
        for comb in self.feature_combinations:
            columns.append(np.prod(X_scaled[:, list(comb)], axis=1))
        X_poly = np.column_stack(columns)
        X_full = self.add_bias(X_poly)
        return X_full @ self.weights

    def score(self, X, y):
        y_pred = self.predict(X)

        y = np.asarray(y)
        if y.ndim == 0 or y.ndim > 2:
            raise ValueError
        if y.ndim == 2:
            if y.shape[0] > 1 and y.shape[1] > 1:
                raise ValueError
            y = y.reshape(-1)

        if y.shape[0] != y_pred.shape[0]:
            raise ValueError

        diff = np.subtract(y, y_pred)
        diff_abs = np.abs(diff)
        diff_square = np.square(diff)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        mae = float(np.mean(diff_abs))
        mse = float(np.mean(diff_square))

        if ss_tot == 0:
            if ss_res == 0:
                r_2 = 1.0
            else:
                r_2 = 0.0
        else:
            r_2 = float(1 - (ss_res / ss_tot))

        return mae, mse, r_2

    def find_optimal_degree(self, X_train, y_train, X_test, y_test, max_degree=15):
        degrees = list(range(1, max_degree + 1))
        train_scores = []
        test_scores = []
        for d in degrees:
            temp = PolynomialRegression(degree=d)
            temp.fit(X_train, y_train)
            mae_train, mse_train, r2_train = temp.score(X_train, y_train)
            mae_test, mse_test, r2_test = temp.score(X_test, y_test)
            train_scores.append(r2_train)
            test_scores.append(r2_test)
        ind = int(np.argmax(test_scores))
        best_degree = degrees[ind]
        best_score = test_scores[ind]
        return {
            "best_degree": best_degree,
            "best_r2": best_score,
            "degrees": degrees,
            "train_r2": train_scores,
            "test_r2": test_scores,
        }

In [5]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# ============================================================
# ТЕСТ 1: create_poly_features для одного признака (degree=2)
# Ожидаем ровно то же, что показано в ноутбуке в ячейке с X_train_poly[:4]
# ============================================================
print("=" * 60)
print("ТЕСТ 1: create_poly_features, один признак, degree=2")
print("=" * 60)
model = PolynomialRegression(degree=2)
model.n_features = 1
X_in = np.array([[110.], [92.], [97.], [150.]])
result = model.create_poly_features(X_in)
print("Вход (X):")
print(X_in.flatten())
print("\nОжидаемо из ноутбука:")
print("[[110, 12100], [92, 8464], [97, 9409], [150, 22500]]")
print("\nПолучено:")
print(result)
print("\nfeature_combinations:", model.feature_combinations)

# ============================================================
# ТЕСТ 2: create_poly_features для двух признаков (degree=2)
# Ожидаем x1, x2, x1², x1*x2, x2²
# ============================================================
print("\n" + "=" * 60)
print("ТЕСТ 2: create_poly_features, два признака, degree=2")
print("=" * 60)
model2 = PolynomialRegression(degree=2)
model2.n_features = 2
X_in2 = np.array([[2., 3.], [4., 5.]])
result2 = model2.create_poly_features(X_in2)
print("Ожидаемо:")
print("[[2, 3, 4, 6, 9], [4, 5, 16, 20, 25]]")
print("\nПолучено:")
print(result2)
print("\nfeature_combinations:", model2.feature_combinations)

# ============================================================
# ТЕСТ 3: scaler работает корректно
# ============================================================
print("\n" + "=" * 60)
print("ТЕСТ 3: fit_scaler + transform_scaler")
print("=" * 60)
X_scaler_test = np.array([[2., 10.], [4., 20.], [6., 30.]])
m = PolynomialRegression()
m.fit_scaler(X_scaler_test)
print("mean:", m.mean)
print("std: ", m.std)
Z = m.transform_scaler(X_scaler_test)
print("\nНормализованные данные (каждый столбец должен иметь mean=0, std=1):")
print(Z)
print("mean по столбцам после нормализации:", Z.mean(axis=0))
print("std по столбцам после нормализации: ", Z.std(axis=0))

ТЕСТ 1: create_poly_features, один признак, degree=2
Вход (X):
[110.  92.  97. 150.]

Ожидаемо из ноутбука:
[[110, 12100], [92, 8464], [97, 9409], [150, 22500]]

Получено:
[[  110. 12100.]
 [   92.  8464.]
 [   97.  9409.]
 [  150. 22500.]]

feature_combinations: [(0,), (0, 0)]

ТЕСТ 2: create_poly_features, два признака, degree=2
Ожидаемо:
[[2, 3, 4, 6, 9], [4, 5, 16, 20, 25]]

Получено:
[[ 2.  3.  4.  6.  9.]
 [ 4.  5. 16. 20. 25.]]

feature_combinations: [(0,), (1,), (0, 0), (0, 1), (1, 1)]

ТЕСТ 3: fit_scaler + transform_scaler
mean: [np.float64(4.0), np.float64(20.0)]
std:  [np.float64(1.632993161855452), np.float64(8.16496580927726)]

Нормализованные данные (каждый столбец должен иметь mean=0, std=1):
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
mean по столбцам после нормализации: [0. 0.]
std по столбцам после нормализации:  [1. 1.]
